# step 1 — RQ2 층 스윕 + K/V 분해

**대응 RQ:** RQ2 — 매개 신호가 형태인가, **어느 층**인가, **어느 경로(Key/Value)**인가. (단일 층 인과는 step C.)

**무엇을 하나 — 두 갈래.**

**A. 개입 측 (인과).** step C의 L25 KV 치환을 **전 층에 반복**하고 각 층에서 세 경로를 분해한다:
**Key만 / Value만 / Key+Value**. 각 (층 × kind)에서 세 상태 `S`(clean/baseline/intervened)와
**회복률** = `(S_int − S_base)/(S_clean − S_base)`를 모두 낸다. donor 2종(`compliant` 주효과 /
`unrelated_camel` 형태 통제)으로 반복 → "내용 아니라 형태" 결론이 전 층에서 유지되는지 본다.

**B. 관측 측 (방향).** 같은 이름의 **camel판 v와 snake판 v의 층별 코사인 유사도** 궤적.
`‖v‖`(크기)는 안 변해도(배율 1.002) 방향이 갈리는지를 본다. 낮으면 "크기 같아도 다른 정보"
→ Value 치환 회복률 피크와 같은 층이면 축 B(Value 경로) 확정.

설계·예측: `docs/step1/plan.md`.

> **메모리(T4):** 개입/코사인 모두 output_attentions 안 씀(KV 캐시 편집·value forward) → eager 불필요, 가볍다.
> 스윕은 선행 프롬프트를 조건당 1회만 forward하고 work 캐시를 층·kind마다 편집→측정→복구로 재사용한다.
> **재개 가능:** 조건마다 저장, 이미 저장된 조건은 건너뜀. GPU 없으면 매우 느림.
> **sanity:** 셀 6에서 S_clean > S_base(위반이 준수 선호를 떨어뜨림) 확인.


In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step1/layer-sweep-kv-split
!git checkout step1/layer-sweep-kv-split
!git pull --quiet origin step1/layer-sweep-kv-split
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — 스윕(전 층 x K/V) 2 donor x 10 seed + 코사인 10 seed. 선행 전부 위반(POOL n=0).
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
DONORS = ['compliant', 'unrelated_camel']   # 음성통제(unrelated_snake)는 step C에서 확인 -> 생략
SEEDS = list(range(10))

def _pre():
    return PrecedingCode(n_compliant=0, n_functions=12, composition=Composition.POOL)
def _ins():
    return Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

def sweep_cond(donor, s):
    # layers='sweep' -> 전 층. kind는 라우팅용(스윕은 내부에서 key/value/key_value 모두 측정).
    return Condition(model=MODEL, preceding=_pre(), instruction=_ins(),
                     intervention=Intervention(kind=InterventionKind.KEY_VALUE,
                                               layers='sweep', donor=donor), seed=s)

def cosine_cond(s):
    return Condition(model=MODEL, preceding=_pre(), instruction=_ins(), seed=s, tag='vcosine')

sweep_conditions = [sweep_cond(d, s) for d in DONORS for s in SEEDS]
cosine_conditions = [cosine_cond(s) for s in SEEDS]

PREDICTION = ('회복률 피크는 후반부(~L25, 상대 0.6~0.75)에 국소화. compliant와 unrelated_camel '
              '곡선은 전 층에서 겹침(형태 결론 유지). Value 단독 회복이 유의하면 축 B 성립. '
              'v 코사인은 피크 층 부근에서 하락 -> Value 회복률 피크와 같은 층이면 축 B 확정.')
print(len(sweep_conditions), '스윕 조건 =', len(DONORS), 'donor x', len(SEEDS), 'seed  (각 조건 = 전 층 x 3 kind)')
print(len(cosine_conditions), '코사인 조건 =', len(SEEDS), 'seed')

In [ ]:
# 실행 — 조건별 즉시 저장(재개). A: 개입 스윕 / B: 코사인 궤적.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

handle = load_model(MODEL)   # output_attentions 안 씀 -> eager 불필요
print('layers:', handle.num_layers,
      '| L25 상대 위치:', round(handle.relative_layer(25), 3),
      '| GQA:', handle.gqa_info())

# A. 개입 스윕 (전 층 x K/V)
new = skipped = 0
for i, c in enumerate(sweep_conditions, 1):
    if result_path(c, step='step1').exists():
        skipped += 1
    else:
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='step1', rq='RQ2', prediction=PREDICTION))
        new += 1
    if i % 4 == 0 or i == len(sweep_conditions):
        print(f'[스윕 {i}/{len(sweep_conditions)}] 새 {new} / 건너뜀 {skipped}')

# B. 코사인 궤적 (관측)
cnew = cskip = 0
for i, c in enumerate(cosine_conditions, 1):
    if result_path(c, step='step1').exists():
        cskip += 1
    else:
        out = run(c, handle=handle, mode='vcosine')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='step1', rq='RQ2', prediction=PREDICTION))
        cnew += 1
print(f'코사인: 새 {cnew} / 건너뜀 {cskip}')
print('완료.')

In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
sweep_recs = [load_result(result_path(c, step='step1')) for c in sweep_conditions]
cosine_recs = [load_result(result_path(c, step='step1')) for c in cosine_conditions]
print('로드: 스윕', len(sweep_recs), '/ 코사인', len(cosine_recs), '-> results/step1/')

In [ ]:
# 요약 — 층별 회복률 곡선(kind x donor) + 피크 층 + v 코사인 궤적. sanity: S_clean > S_base.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from harness.intervention import peak_layer

KINDS = ['key', 'value', 'key_value']

# 층별 회복률: donor -> kind -> {layer: [seed별 회복률]}
agg = {d: {k: defaultdict(list) for k in KINDS} for d in DONORS}
for r in sweep_recs:
    d = r.condition.intervention.donor
    for L, flat in r.metrics.per_layer.items():
        for k in KINDS:
            key = f'{k}__recovery'
            if key in flat:
                agg[d][k][int(L)].append(flat[key])

def curve(d, k):
    layers = sorted(agg[d][k])
    return layers, [float(np.mean(agg[d][k][L])) for L in layers]

# 피크 층 표
print('=== 피크 층 (회복률 최대) ===')
rows = []
for d in DONORS:
    for k in KINDS:
        layers, vals = curve(d, k)
        pk = peak_layer(dict(zip(layers, vals)))
        rows.append({'donor': d, 'kind': k, 'peak_layer': pk[0], 'peak_recovery': round(pk[1], 3)})
print(pd.DataFrame(rows).to_string(index=False))

# v 코사인 궤적 평균
cos = defaultdict(list)
for r in cosine_recs:
    for L, flat in r.metrics.per_layer.items():
        cos[int(L)].append(flat['v_cosine'])
clayers = sorted(cos); cvals = [float(np.mean(cos[L])) for L in clayers]

# 플롯: donor별 회복률 곡선 2 + 코사인 궤적 1
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
colors = {'key': '#2563C9', 'value': '#C6771A', 'key_value': '#2E7D52'}
for j, d in enumerate(DONORS):
    for k in KINDS:
        layers, vals = curve(d, k)
        ax[j].plot(layers, vals, color=colors[k], label=k, marker='.', ms=4)
    ax[j].axhline(0, color='#aaa', lw=.6); ax[j].axhline(1, color='#aaa', ls='--', lw=.6)
    ax[j].axvline(25, color='#B0392B', ls=':', lw=.8)
    ax[j].set_title(f'Recovery sweep — donor={d}')
    ax[j].set_xlabel('layer'); ax[j].set_ylabel('recovery'); ax[j].legend(fontsize=8)
ax[2].plot(clayers, cvals, color='#7B3FA0', marker='.', ms=4)
ax[2].axvline(25, color='#B0392B', ls=':', lw=.8)
ax[2].set_title('v cosine trajectory (camel vs snake)')
ax[2].set_xlabel('layer'); ax[2].set_ylabel('cosine (direction)'); ax[2].set_ylim(-0.1, 1.05)
plt.tight_layout(); plt.savefig('step1_summary.png', dpi=110); plt.show()

sc = float(np.mean([r.metrics.extra['S_clean'] for r in sweep_recs]))
sb = float(np.mean([r.metrics.extra['S_base'] for r in sweep_recs]))
print(f'sanity  S_clean {sc:+.2f} > S_base {sb:+.2f} :', sc > sb)

In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('step1_results', 'zip', 'results/step1')
try:
    from google.colab import files
    files.download('step1_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): step1_results.zip', e)